## ML_1024: KV Cache Attention

**Difficulty**: Hard | **TorchCode**: #14

### Problem
Implement multi-head attention with a KV cache for efficient autoregressive decoding. During prefill, apply a causal mask. During decode, append new K/V to the cache and attend over the full history.

### Formula
$$K_{\text{total}} = [K_{\text{cache}};\, K_{\text{new}}], \quad \text{out}, \text{cache} = \text{MHA}(Q_{\text{new}}, K_{\text{total}}, V_{\text{total}})$$

In [ ]:
import torch
import torch.nn as nn
import math

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        B, S_new, _ = x.shape
        q = self.W_q(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        new_cache = (k, v)
        S_total = k.shape[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if S_new > 1:
            S_past = S_total - S_new
            mask = torch.triu(
                torch.ones(S_new, S_total, device=x.device, dtype=torch.bool),
                diagonal=S_past + 1,
            )
            scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        out = self.W_o(attn.transpose(1, 2).contiguous().view(B, S_new, -1))
        return out, new_cache

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape (no cache) ---
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
assert isinstance(attn, nn.Module)
out, cache = attn(torch.randn(2, 8, 64))
assert out.shape == (2, 8, 64)

# --- Test 2: Cache structure ---
torch.manual_seed(0)
attn2 = KVCacheAttention(d_model=64, num_heads=4)
out, cache = attn2(torch.randn(2, 8, 64))
assert isinstance(cache, tuple) and len(cache) == 2
assert cache[0].shape == (2, 4, 8, 16)
assert cache[1].shape == (2, 4, 8, 16)

# --- Test 3: Decode step appends to cache ---
torch.manual_seed(0)
attn3 = KVCacheAttention(d_model=32, num_heads=2)
_, cache = attn3(torch.randn(1, 4, 32))
out, new_cache = attn3(torch.randn(1, 1, 32), cache=cache)
assert out.shape == (1, 1, 32)
assert new_cache[0].shape[2] == 5

# --- Test 4: Incremental decode matches full forward ---
torch.manual_seed(42)
attn4 = KVCacheAttention(d_model=32, num_heads=2)
x = torch.randn(1, 6, 32)
full_out, _ = attn4(x)
out1, cache = attn4(x[:, :4])
out2, cache = attn4(x[:, 4:5], cache=cache)
out3, cache = attn4(x[:, 5:6], cache=cache)
assert torch.allclose(full_out, torch.cat([out1, out2, out3], dim=1), atol=1e-5)

# --- Test 5: Gradient flow ---
torch.manual_seed(0)
attn5 = KVCacheAttention(d_model=32, num_heads=2)
x = torch.randn(1, 4, 32, requires_grad=True)
attn5(x)[0].sum().backward()
assert x.grad is not None

print('All tests passed!')